IMPORT

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

BASE_DIR = Path("..")

CLEANED_DATA_PATH = BASE_DIR / "data" / "processed" / "cleaned_online_retail.csv"
RFM_PATH = BASE_DIR / "data" / "processed" / "rfm_features.csv"

PROCESSED_DIR = BASE_DIR / "data" / "processed"
TABLES_DIR = BASE_DIR / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

Setup complete.


LOAD DATA

In [2]:
df = pd.read_csv(CLEANED_DATA_PATH)
rfm = pd.read_csv(RFM_PATH)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Transaction data:", df.shape)
print("RFM data:", rfm.shape)

display(df.head())
display(rfm.head())

Transaction data: (392692, 9)
RFM data: (4338, 4)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


CREATE INVOICE LEVEL CUSTOMER DATA

In [3]:
invoice_df = (
    df.groupby(["CustomerID", "InvoiceNo"])
    .agg(
        InvoiceDate=("InvoiceDate", "min"),
        InvoiceValue=("TotalPrice", "sum"),
        TotalQuantity=("Quantity", "sum"),
        UniqueProducts=("StockCode", "nunique")
    )
    .reset_index()
)

invoice_df = invoice_df.sort_values(["CustomerID", "InvoiceDate"])

print("Invoice-level data:", invoice_df.shape)
display(invoice_df.head())

Invoice-level data: (18532, 6)


,CustomerID,InvoiceNo,InvoiceDate,InvoiceValue,TotalQuantity,UniqueProducts
0,12346,541431,2011-01-18 10:01:00,77183.60,74215,1
1,12347,537626,2010-12-07 14:57:00,711.79,319,31
2,12347,542237,2011-01-26 14:30:00,475.39,315,29
3,12347,549222,2011-04-07 10:43:00,636.25,483,24
4,12347,556201,2011-06-09 13:01:00,382.52,196,18


VELOCITY ADN CUSTOMER LIFETIME FEATURE

In [4]:
customer_time = (
    invoice_df.groupby("CustomerID")
    .agg(
        FirstPurchase=("InvoiceDate", "min"),
        LastPurchase=("InvoiceDate", "max"),
        InvoiceCount=("InvoiceNo", "nunique"),
        TotalSpend=("InvoiceValue", "sum")
    )
    .reset_index()
)

customer_time["Customer_Lifetime_Days"] = (
    customer_time["LastPurchase"] - customer_time["FirstPurchase"]
).dt.days + 1

customer_time["Purchase_Velocity"] = (
    customer_time["InvoiceCount"] / customer_time["Customer_Lifetime_Days"]
)

customer_time["Spend_Velocity"] = (
    customer_time["TotalSpend"] / customer_time["Customer_Lifetime_Days"]
)

customer_time = customer_time[
    [
        "CustomerID",
        "Customer_Lifetime_Days",
        "Purchase_Velocity",
        "Spend_Velocity"
    ]
]

display(customer_time.head())

,CustomerID,Customer_Lifetime_Days,Purchase_Velocity,Spend_Velocity
0,12346,1,1.000000,77183.600000
1,12347,366,0.019126,11.775956
2,12348,283,0.014134,6.350671
3,12349,1,1.000000,1757.550000
4,12350,1,1.000000,334.400000


PURCHASE INTERVAL VOLATILITY

In [5]:
invoice_df["Purchase_Interval"] = (
    invoice_df.groupby("CustomerID")["InvoiceDate"]
    .diff()
    .dt.days
)

interval_features = (
    invoice_df.groupby("CustomerID")
    .agg(
        Purchase_Interval_Mean=("Purchase_Interval", "mean"),
        Purchase_Interval_Std=("Purchase_Interval", "std"),
        Purchase_Interval_Max=("Purchase_Interval", "max")
    )
    .fillna(0)
    .reset_index()
)

interval_features["Purchase_Regularity_Index"] = 1 / (
    1 + interval_features["Purchase_Interval_Std"]
)

display(interval_features.head())

,CustomerID,Purchase_Interval_Mean,Purchase_Interval_Std,Purchase_Interval_Max,Purchase_Regularity_Index
0,12346,0.000000,0.000000,0.0,1.000000
1,12347,60.333333,18.478817,90.0,0.051338
2,12348,94.000000,70.149840,173.0,0.014055
3,12349,0.000000,0.000000,0.0,1.000000
4,12350,0.000000,0.000000,0.0,1.000000


SPENDING VOLATITLTY

In [6]:
spending_features = (
    invoice_df.groupby("CustomerID")
    .agg(
        Invoice_Value_Mean=("InvoiceValue", "mean"),
        Invoice_Value_Std=("InvoiceValue", "std"),
        Invoice_Value_Max=("InvoiceValue", "max"),
        Invoice_Value_Min=("InvoiceValue", "min")
    )
    .fillna(0)
    .reset_index()
)

spending_features["Invoice_Value_CV"] = (
    spending_features["Invoice_Value_Std"] /
    spending_features["Invoice_Value_Mean"].replace(0, np.nan)
).fillna(0)

spending_features["Invoice_Value_Range"] = (
    spending_features["Invoice_Value_Max"] -
    spending_features["Invoice_Value_Min"]
)

display(spending_features.head())

,CustomerID,Invoice_Value_Mean,Invoice_Value_Std,Invoice_Value_Max,Invoice_Value_Min,Invoice_Value_CV,Invoice_Value_Range
0,12346,77183.600000,0.000000,77183.60,77183.60,0.000000,0.00
1,12347,615.714286,341.070789,1294.32,224.82,0.553943,1069.50
2,12348,449.310000,301.159918,892.80,227.44,0.670272,665.36
3,12349,1757.550000,0.000000,1757.55,1757.55,0.000000,0.00
4,12350,334.400000,0.000000,334.40,334.40,0.000000,0.00


BEHAVIOURAL MOMENTUM

In [7]:
momentum_rows = []

for customer_id, group in invoice_df.groupby("CustomerID"):
    group = group.sort_values("InvoiceDate")

    values = group["InvoiceValue"].values

    if len(values) >= 2:
        midpoint = len(values) // 2
        first_half_mean = values[:midpoint].mean()
        second_half_mean = values[midpoint:].mean()

        spending_momentum = second_half_mean - first_half_mean
        spending_momentum_ratio = second_half_mean / first_half_mean if first_half_mean != 0 else 0
    else:
        spending_momentum = 0
        spending_momentum_ratio = 0

    momentum_rows.append({
        "CustomerID": customer_id,
        "Spending_Momentum": spending_momentum,
        "Spending_Momentum_Ratio": spending_momentum_ratio
    })

momentum_features = pd.DataFrame(momentum_rows)

display(momentum_features.head())

,CustomerID,Spending_Momentum,Spending_Momentum_Ratio
0,12346,0.0000,0.000000
1,12347,13.8325,1.022758
2,12348,-221.6200,0.604335
3,12349,0.0000,0.000000
4,12350,0.0000,0.000000


PRODUCT DIVERSITY FEATURES

In [8]:
product_features = (
    invoice_df.groupby("CustomerID")
    .agg(
        Avg_Products_Per_Invoice=("UniqueProducts", "mean"),
        Std_Products_Per_Invoice=("UniqueProducts", "std"),
        Avg_Quantity_Per_Invoice=("TotalQuantity", "mean")
    )
    .fillna(0)
    .reset_index()
)

display(product_features.head())

,CustomerID,Avg_Products_Per_Invoice,Std_Products_Per_Invoice,Avg_Quantity_Per_Invoice
0,12346,1.00,0.000000,74215.000000
1,12347,26.00,11.430952,351.142857
2,12348,6.75,4.349329,585.250000
3,12349,73.00,0.000000,631.000000
4,12350,17.00,0.000000,197.000000


MERGE ADVANCE BEHAVIOURAL FEATURES

In [9]:
advanced_features = (
    rfm
    .merge(customer_time, on="CustomerID", how="left")
    .merge(interval_features, on="CustomerID", how="left")
    .merge(spending_features, on="CustomerID", how="left")
    .merge(momentum_features, on="CustomerID", how="left")
    .merge(product_features, on="CustomerID", how="left")
)

advanced_features = advanced_features.fillna(0)

print("Advanced feature dataset:", advanced_features.shape)
display(advanced_features.head())

Advanced feature dataset: (4338, 22)


,CustomerID,Recency,Frequency,Monetary,Customer_Lifetime_Days,Purchase_Velocity,Spend_Velocity,Purchase_Interval_Mean,Purchase_Interval_Std,Purchase_Interval_Max,...,Invoice_Value_Std,Invoice_Value_Max,Invoice_Value_Min,Invoice_Value_CV,Invoice_Value_Range,Spending_Momentum,Spending_Momentum_Ratio,Avg_Products_Per_Invoice,Std_Products_Per_Invoice,Avg_Quantity_Per_Invoice
0,12346,326,1,77183.60,1,1.000000,77183.600000,0.000000,0.000000,0.0,...,0.000000,77183.60,77183.60,0.000000,0.00,0.0000,0.000000,1.00,0.000000,74215.000000
1,12347,2,7,4310.00,366,0.019126,11.775956,60.333333,18.478817,90.0,...,341.070789,1294.32,224.82,0.553943,1069.50,13.8325,1.022758,26.00,11.430952,351.142857
2,12348,75,4,1797.24,283,0.014134,6.350671,94.000000,70.149840,173.0,...,301.159918,892.80,227.44,0.670272,665.36,-221.6200,0.604335,6.75,4.349329,585.250000
3,12349,19,1,1757.55,1,1.000000,1757.550000,0.000000,0.000000,0.0,...,0.000000,1757.55,1757.55,0.000000,0.00,0.0000,0.000000,73.00,0.000000,631.000000
4,12350,310,1,334.40,1,1.000000,334.400000,0.000000,0.000000,0.0,...,0.000000,334.40,334.40,0.000000,0.00,0.0000,0.000000,17.00,0.000000,197.000000


REFINED BEHAVIOURAL FEATURE DATASET

In [ ]:


refined_advanced_features = advanced_features[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "Customer_Lifetime_Days",
        "Purchase_Velocity",
        "Purchase_Interval_Std",
        "Purchase_Regularity_Index",
        "Invoice_Value_CV"
    ]
]

print("Refined advanced feature dataset shape:")
print(refined_advanced_features.shape)

display(refined_advanced_features.head())

Refined advanced feature dataset shape:
(4338, 9)


,CustomerID,Recency,Frequency,Monetary,Customer_Lifetime_Days,Purchase_Velocity,Purchase_Interval_Std,Purchase_Regularity_Index,Invoice_Value_CV
0,12346,326,1,77183.60,1,1.000000,0.000000,1.000000,0.000000
1,12347,2,7,4310.00,366,0.019126,18.478817,0.051338,0.553943
2,12348,75,4,1797.24,283,0.014134,70.149840,0.014055,0.670272
3,12349,19,1,1757.55,1,1.000000,0.000000,1.000000,0.000000
4,12350,310,1,334.40,1,1.000000,0.000000,1.000000,0.000000


SAVE REFINED DATASET

In [ ]:

refined_advanced_features.to_csv(
    PROCESSED_DIR / "rfm_refined_advanced_features.csv",
    index=False
)

print("Saved successfully:")
print(PROCESSED_DIR / "rfm_refined_advanced_features.csv")

Saved successfully:
..\data\processed\rfm_refined_advanced_features.csv


SAVE ADVANCE FEATURES

In [10]:
advanced_features.to_csv(
    PROCESSED_DIR / "rfm_advanced_behavioural_features.csv",
    index=False
)

advanced_features.describe().T.to_csv(
    TABLES_DIR / "advanced_behavioural_feature_summary.csv"
)

print("Saved:")
print(PROCESSED_DIR / "rfm_advanced_behavioural_features.csv")
print(TABLES_DIR / "advanced_behavioural_feature_summary.csv")

Saved:
..\data\processed\rfm_advanced_behavioural_features.csv
..\outputs\tables\advanced_behavioural_feature_summary.csv
